# Phase 5 — Evaluation, Analysis & Demo

Unified evaluation notebook for all four medical text simplification systems.

**Systems**: rule_based · t5_small · scifive · hybrid  
**Metrics**: SARI · BLEU · FKGL  
**Test set**: 1,046 sentence pairs (PLABA + Cochrane)

---

**Run all cells top-to-bottom. No Google Drive or GPU required.**

## Section 1 — Setup

In [ ]:
import importlib, subprocess, sys

def _install(pkg, import_name=None):
    name = import_name or pkg
    try:
        importlib.import_module(name)
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

_install('matplotlib')
_install('pandas')
_install('textstat')
_install('sacrebleu')
_install('easse @ git+https://github.com/feralvam/easse.git', 'easse')

print('Dependencies ready.')

In [ ]:
# Colab setup: clones full repo so predictions/samples/ and all data files exist.
# On local Jupyter this cell is skipped automatically.
try:
    import google.colab  # only importable inside Colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    import subprocess, os
    REPO_URL = "https://github.com/IbrahimHanafy2222/NLP-Project-Submission"
    CLONE_DIR = "/content/NLP-Project-Submission"
    if not os.path.exists(CLONE_DIR):
        print("Cloning repo...")
        subprocess.check_call(["git", "clone", REPO_URL, CLONE_DIR])
    os.chdir(CLONE_DIR)
    print(f"Colab ready. CWD = {os.getcwd()}")
else:
    print("Local environment — skipped repo clone.")


In [ ]:
import os, sys, json, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

matplotlib.rcParams.update({
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.labelsize': 11,
})

import pathlib
REPO_ROOT = str(pathlib.Path(os.path.abspath('.')).parent if
                os.path.basename(os.getcwd()) == 'notebooks' else
                os.path.abspath('.'))
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)
os.chdir(REPO_ROOT)

print('Seed set:', RANDOM_SEED)
print('Repo root:', REPO_ROOT)

## Section 2 — Load Predictions (Sample Set)

In [ ]:
SYSTEMS     = ['rule_based', 't5_small', 'scifive', 'hybrid']
SAMPLES_DIR = os.path.join(REPO_ROOT, 'predictions', 'samples')

print(f'Loading predictions from: {SAMPLES_DIR}')
assert os.path.isdir(SAMPLES_DIR), (
    f'predictions/samples/ not found at {SAMPLES_DIR}.\n'
    'If on Colab: close tab, re-open fresh from GitHub (File > Open notebook > GitHub tab).'
)

predictions = {}
for system in SYSTEMS:
    path = os.path.join(SAMPLES_DIR, f'{system}.jsonl')
    with open(path, encoding='utf-8') as f:
        records = [json.loads(line) for line in f]
    assert len(records) > 0, f'{system}: sample file is empty'
    assert all('source' in r and 'prediction' in r and 'reference' in r for r in records)
    predictions[system] = records
    print(f'{system}: {len(records)} sample records loaded')

print('\nSample (rule_based[0]):')
r = predictions['rule_based'][0]
print(f'  SOURCE:     {r["source"][:80]}')
print(f'  PREDICTION: {r["prediction"][:80]}')
print(f'  REFERENCE:  {r["reference"][:80]}')

## Section 3 — Results Table

In [ ]:
df = pd.read_csv('results/metrics.csv')

DISPLAY_ORDER = ['rule_based', 't5_small', 'scifive', 'hybrid']
df_display = (
    df.set_index('system')
      .reindex([s for s in DISPLAY_ORDER if s in df['system'].values])
      .reset_index()
)

print('=' * 72)
print('RESULTS: All Systems × All Metrics')
print('=' * 72)
print(df_display.to_string(index=False))
print('=' * 72)

t5_sari     = df.loc[df['system'] == 't5_small', 'sari'].values[0]
hybrid_sari = df.loc[df['system'] == 'hybrid',   'sari'].values[0]
delta       = hybrid_sari - t5_sari
direction   = 'improvement' if delta > 0 else 'regression'
print(f'\nHybrid vs T5-small SARI: {delta:+.4f} ({direction})')
if delta <= 0:
    print('Note: Hybrid underperforms T5-small on SARI — reported honestly.')

## Section 4 — FKGL Bar Chart

In [ ]:
os.makedirs('results/figures', exist_ok=True)

systems_ordered = ['rule_based', 't5_small', 'scifive', 'hybrid']
labels          = ['Rule-Based', 'T5-Small', 'SciFive', 'Hybrid']

fig, ax = plt.subplots(figsize=(8, 4))
deltas  = [df.loc[df['system'] == s, 'fkgl_delta'].values[0] for s in systems_ordered]
colors  = ['#5B9BD5' if d < -3 else '#ED7D31' for d in deltas]

bars = ax.barh(labels, deltas, color=colors, edgecolor='white', height=0.5)
ax.axvline(0, color='black', linewidth=0.8, linestyle='--')
for bar, val in zip(bars, deltas):
    ax.text(val - 0.05, bar.get_y() + bar.get_height() / 2,
            f'{val:.2f}', va='center', ha='right', fontsize=10, fontweight='bold')

ax.set_xlabel('FKGL Delta (output − input; negative = simpler)')
ax.set_title('Readability Change by System')
ax.invert_yaxis()
plt.tight_layout()
fig.savefig('results/figures/fkgl_bar.png', dpi=300, bbox_inches='tight')
plt.show()
print('Saved results/figures/fkgl_bar.png')

## Section 5 — SARI Bar Chart

In [ ]:
fig, ax    = plt.subplots(figsize=(8, 4))
saris      = [df.loc[df['system'] == s, 'sari'].values[0] for s in systems_ordered]
colors_sari = ['#5B9BD5' if v > 20 else '#ED7D31' for v in saris]

bars = ax.barh(labels, saris, color=colors_sari, edgecolor='white', height=0.5)
ax.axvline(0, color='black', linewidth=0.8)
for bar, val in zip(bars, saris):
    ax.text(val - 0.3, bar.get_y() + bar.get_height() / 2,
            f'{val:.2f}', va='center', ha='right', fontsize=10,
            fontweight='bold', color='white')

ax.set_xlabel('SARI Score (higher = better simplification)')
ax.set_title('SARI Score by System')
ax.invert_yaxis()
plt.tight_layout()
fig.savefig('results/figures/sari_bar.png', dpi=300, bbox_inches='tight')
plt.show()
print('Saved results/figures/sari_bar.png')

## Section 6 — Qualitative Example Matrix

In [ ]:
IDX = 0
source    = predictions['rule_based'][IDX]['source']
reference = predictions['rule_based'][IDX]['reference']

print('=' * 80)
print('QUALITATIVE EXAMPLE MATRIX')
print('=' * 80)
print(f'SOURCE:      {source}')
print(f'REFERENCE:   {reference}')
print('-' * 80)
for system, label in zip(systems_ordered, labels):
    pred = predictions[system][IDX]['prediction']
    print(f'{label:<12}: {pred}')
print('=' * 80)

## Section 7 — Error Analysis

In [ ]:
def assign_failure_mode(source: str, prediction: str, reference: str) -> str:
    pred_clean = prediction.strip()
    src_clean  = source.strip()
    if pred_clean == src_clean:
        return 'no-change'
    if len(pred_clean.split()) < 0.4 * len(src_clean.split()):
        return 'over-simplification'
    if not pred_clean.endswith(('.', '?', '!')):
        return 'incomplete'
    return 'hallucination'

def find_failure_examples(records, n=2):
    examples = []
    for r in records:
        mode = assign_failure_mode(r['source'], r['prediction'], r['reference'])
        examples.append({**r, 'failure_mode': mode})
        if len(examples) >= n:
            break
    return examples

print('Failure mode heuristics defined.')

In [ ]:
FAILURE_MODES = {'no-change', 'over-simplification', 'incomplete', 'hallucination'}

print('=' * 80)
print('PER-SYSTEM ERROR ANALYSIS (2 examples per system)')
print('=' * 80)

all_failures = {}
for system, label in zip(systems_ordered, labels):
    examples = find_failure_examples(predictions[system], n=2)
    all_failures[system] = examples
    print(f'\n--- {label} ---')
    for i, ex in enumerate(examples, 1):
        print(f'  [{i}] failure_mode: {ex["failure_mode"]}')
        print(f'      SOURCE:     {ex["source"][:100]}')
        print(f'      PREDICTION: {ex["prediction"][:100]}')
        print(f'      REFERENCE:  {ex["reference"][:100]}')

print('\n' + '=' * 80)

## Section 8 — Assertions

In [ ]:
df_verify = pd.read_csv('results/metrics.csv')
assert len(df_verify) == 4
assert set(df_verify['system']) == {'rule_based', 't5_small', 'scifive', 'hybrid'}
assert all(df_verify['fkgl_delta'] < 0)
assert os.path.exists('results/figures/fkgl_bar.png')
assert os.path.exists('results/figures/sari_bar.png')
for system in SYSTEMS:
    assert len(all_failures[system]) == 2
    for ex in all_failures[system]:
        assert ex['failure_mode'] in FAILURE_MODES

print('ALL ASSERTIONS PASSED')
print(f'  metrics.csv: 4 rows, all fkgl_delta < 0')
print(f'  figures: fkgl_bar.png, sari_bar.png')
print(f'  error analysis: 4 systems × 2 failure examples')